Purpose: Train deberta so I don't have to train every time I run 

In [1]:
### Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import torch

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from torch.utils.data import Dataset
import accelerate

In [2]:
SEED=42
MODEL_NAME='microsoft/deberta-v3-base'

MAX_LEN=128
BATCH_SIZE=16
NUM_EPOCHS=5
LR=2e-5

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

ANNOTATED_FILE = Path("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample_annotated.csv")

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = OUTPUT_DIR/"deberta_classifier"

custom_dim = ['Narrative Structure & Quality', 'Character & Emotion', 'Originality', 'Immersion', 'Thematic Depth', 'Writing Style']

In [3]:
ann_df = pd.read_csv(ANNOTATED_FILE)
ann_df = ann_df.dropna(subset=['sentence'])

print(ann_df.shape)
ann_df.head()

(3000, 10)


,review_id,sentence_idx,sentence,language,Narrative Structure & Quality,Character & Emotion,Originality,Immersion,Thematic Depth,Writing Style
0,6c98fe733ae0c27671ebbbe68b77fd8f,6,The initial deepening of the mechanics of the ...,eng,0,0,0,1,0,0
1,cd8abbbf2727515f904a6b189cb0eb84,24,I think that's more troubling when it comes to...,eng,0,0,0,0,0,0
2,b0d8887563f48d59440cd78144ef23c0,59,The one note simple tone of everything leads m...,en-US,0,0,0,0,0,1
3,93060ddc1ef84b111915ed91cfc443de,35,Saving grace was that he was the only characte...,eng,0,1,0,0,0,0
4,4fac8a39591a791eb0a78c4c08176d2b,4,I've never been so disappointed by this author.,eng,0,0,0,0,0,0


In [15]:
df=ann_df.dropna(subset=["sentence"])
labels=df[custom_dim].astype(np.float32).values
texts=df["sentence"].tolist()
print(df.shape)
print(labels.mean(axis=0))

(3000, 10)
[0.16       0.19266666 0.03566667 0.034      0.02933333 0.06266667]


In [4]:
class SentenceDataset(Dataset):
    def __init__(self,texts, labels, tokenizer):

        self.enc = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN,return_tensors='pt')
        self.labels=torch.tensor(labels,dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self,idx):
        return {
            'input_ids':
            self.enc['input_ids'][idx],

            'attention_mask':
            self.enc['attention_mask'][idx],

            'labels':
            self.labels[idx]
        }

In [14]:
def build_compute_metrics(threshold: float = 0.5):

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        
        print("compute_metrics called")
        print(logits.shape)
        print(labels.shape)

        probs = 1 / (1 + np.exp(-logits))  # sigmoid
        preds = (probs >= threshold).astype(int)

        micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        per_dim  = f1_score(labels, preds, average=None, zero_division=0)

        metrics = {'f1_micro': micro_f1, 'f1_macro': macro_f1}
        for dim, score in zip(custom_dim, per_dim):
            metrics[f'f1_{dim}'] = score
        return metrics
    return compute_metrics


def train_classifier( ann_df: pd.DataFrame, model_dir: Path = MODEL_DIR, test_size: float = 0.15, val_size: float = 0.15,):

    sentences = ann_df['sentence'].tolist()
    labels    = ann_df[custom_dim].values.astype(np.float32)

    # Split: train / val / test
    idx = np.arange(len(sentences))
    idx_trainval, idx_test = train_test_split(idx, test_size=test_size, random_state=SEED)
    idx_train, idx_val     = train_test_split(idx_trainval, test_size=val_size, random_state=SEED)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_ds = SentenceDataset([sentences[i] for i in idx_train], labels[idx_train], tokenizer)
    val_ds   = SentenceDataset([sentences[i] for i in idx_val],   labels[idx_val],   tokenizer)
    test_ds  = SentenceDataset([sentences[i] for i in idx_test],  labels[idx_test],  tokenizer)

    for i in range(len(train_ds)):
        lbl = train_ds[i]['labels']
        if lbl.sum() > 0:
            print("Found positive example:", i)
            print(lbl)
            break

    print("Train",len(train_ds)) 
    print("Val",len(val_ds))
    print("Test",len(test_ds))

    sample = train_ds[i]

    print(sample['input_ids'][:20])
    print(sample['attention_mask'][:20])

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(custom_dim),
        problem_type='multi_label_classification',  # triggers BCE + sigmoid internally
    )

    model = model.to(DEVICE)

    training_args = TrainingArguments(
        output_dir=str(model_dir),

        num_train_epochs=NUM_EPOCHS,

        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,

        learning_rate=LR,
        weight_decay=0.01,

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,

        metric_for_best_model="f1_macro",
        greater_is_better=True,

        report_to="none",

        label_names=["labels"],      # <-- ADD THIS

        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=build_compute_metrics(),
    )

    batch = next(iter(trainer.get_train_dataloader()))

    print(batch.keys())
    print(batch['labels'].shape)
    print(batch['labels'].dtype)
    print(batch['labels'][0])

    batch = train_ds[0]

    device = next(model.parameters()).device

    inputs = {
        'input_ids': batch['input_ids'].unsqueeze(0).to(device),
        'attention_mask': batch['attention_mask'].unsqueeze(0).to(device),
        'labels': batch['labels'].unsqueeze(0).to(device)
    }

    with torch.no_grad():
        outputs = model(**inputs)

    print("LOSS:", outputs.loss)
    print("LOGITS:", outputs.logits)

    trainer.train()
    print(trainer.state.log_history)
    tokenizer.save_pretrained(model_dir)

    # Final evaluation on held-out test set
    test_results = trainer.evaluate(test_ds)
    print('\n── Test-set evaluation ──')
    for k, v in test_results.items():
        print(f'  {k}: {v:.4f}')

    return trainer

#labels = ann_df[custom_dim].values.astype(np.float32)
trainer=train_classifier(ann_df)

Found positive example: 1
tensor([1., 0., 0., 0., 0., 0.])
Train 2167
Val 383
Test 450
tensor([    1,   279,   697,   284,  1274,   271, 16868,   263,   343,   284,
          298,   266,  6715,   465,   272,   273,   295,   428,   265,   260])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([16, 6])
torch.float32
tensor([0., 0., 0., 0., 0., 0.], device='cuda:0')
LOSS: tensor(0.6610, device='cuda:0')
LOGITS: tensor([[-0.2054, -0.0583, -0.2148, -0.0189,  0.0230,  0.0650]],
       device='cuda:0', dtype=torch.float16)


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Narrative structure & quality,F1 Character & emotion,F1 Originality,F1 Immersion,F1 Thematic depth,F1 Writing style
1,162.234820,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[{'loss': 162.2348201976103, 'grad_norm': nan, 'learning_rate': 1.6029411764705884e-05, 'epoch': 1.0, 'step': 136}, {'eval_loss': nan, 'eval_f1_micro': 0.0, 'eval_f1_macro': 0.0, 'eval_f1_Narrative Structure & Quality': 0.0, 'eval_f1_Character & Emotion': 0.0, 'eval_f1_Originality': 0.0, 'eval_f1_Immersion': 0.0, 'eval_f1_Thematic Depth': 0.0, 'eval_f1_Writing Style': 0.0, 'eval_runtime': 0.4682, 'eval_samples_per_second': 818.074, 'eval_steps_per_second': 51.263, 'epoch': 1.0, 'step': 136}, {'loss': 0.0, 'grad_norm': nan, 'learning_rate': 1.2029411764705882e-05, 'epoch': 2.0, 'step': 272}, {'eval_loss': nan, 'eval_f1_micro': 0.0, 'eval_f1_macro': 0.0, 'eval_f1_Narrative Structure & Quality': 0.0, 'eval_f1_Character & Emotion': 0.0, 'eval_f1_Originality': 0.0, 'eval_f1_Immersion': 0.0, 'eval_f1_Thematic Depth': 0.0, 'eval_f1_Writing Style': 0.0, 'eval_runtime': 0.4687, 'eval_samples_per_second': 817.156, 'eval_steps_per_second': 51.206, 'epoch': 2.0, 'step': 272}, {'loss': 0.0, 'grad_n

compute_metrics called
(450, 6)
(450, 6)


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,F1 Narrative structure & quality,F1 Character & emotion,F1 Originality,F1 Immersion,F1 Thematic depth,F1 Writing style
0.000000,nan,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



── Test-set evaluation ──
  eval_loss: nan
  eval_f1_micro: 0.0000
  eval_f1_macro: 0.0000
  eval_f1_Narrative Structure & Quality: 0.0000
  eval_f1_Character & Emotion: 0.0000
  eval_f1_Originality: 0.0000
  eval_f1_Immersion: 0.0000
  eval_f1_Thematic Depth: 0.0000
  eval_f1_Writing Style: 0.0000


In [6]:
ann_df[custom_dim].dtypes

Narrative Structure & Quality    int64
Character & Emotion              int64
Originality                      int64
Immersion                        int64
Thematic Depth                   int64
Writing Style                    int64
dtype: object

### Troubleshoot

In [7]:
ann_df[custom_dim].head(10)

,Narrative Structure & Quality,Character & Emotion,Originality,Immersion,Thematic Depth,Writing Style
0,0,0,0,1,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,1
3,0,1,0,0,0,0
4,0,0,0,0,0,0
5,1,1,0,0,0,0
6,0,0,0,0,0,0
7,0,0,0,0,0,0
8,0,0,0,0,0,0
9,0,0,0,0,0,0


In [8]:
ann_df[custom_dim].describe()

,Narrative Structure & Quality,Character & Emotion,Originality,Immersion,Thematic Depth,Writing Style
count,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,0.160000,0.192667,0.035667,0.034000,0.029333,0.062667
std,0.366667,0.394459,0.185489,0.181259,0.168767,0.242403
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
import transformers
import inspect
print(transformers.__version__)
print(inspect.signature(TrainingArguments))

5.12.1
(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_c

In [10]:
import sys
print(sys.executable)

/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/venv/bin/python


In [11]:
import accelerate
print(accelerate.__version__)

1.14.0


In [12]:
print(torch.__version__)
print(transformers.__version__)
print(accelerate.__version__)

2.12.1+cu130
5.12.1
1.14.0
